# ForensiX: get started in Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anas-rz/ForensiX/blob/main/notebooks/ForensiX_Get_Started.ipynb)

Detect, calibrate, evaluate and serve deepfake detectors on a free Colab GPU. Every model and
every sample file here is under a commercially usable license.

| Section | What you'll do | Time on a T4 |
|---|---|---|
| 1. Setup | Install ForensiX | 2 min |
| 2. Sample media | Download real and AI-generated images, speech and face-swap videos | 2 min |
| 3. Detect | Score images, audio and video; check C2PA credentials and watermarks; upload your own files | 3 min |
| 4. Calibrate | Fit a calibration on labelled videos, then get calibrated probabilities and confidence verdicts | 3 min |
| 5. Evaluate | Measure detectors on held-out data: AUC, EER, false alarms, detection rate | 10 min |
| 6. Web manager | Run the HTTP server and open the web manager inside Colab; optional fairness report | 3 min |
| 7. Keep your results | Save calibrations and results to Google Drive | 1 min |

**Before you start:** Runtime → Change runtime type → **T4 GPU** (or better). Everything also runs
on CPU, much more slowly for video.

Full documentation: [https://anas-rz.github.io/ForensiX/](https://anas-rz.github.io/ForensiX/)

## 1. Setup

In [ ]:
!nvidia-smi -L || echo "No GPU. Runtime -> Change runtime type -> T4 GPU (video models are slow on CPU)"
!if [ ! -d /content/ForensiX ]; then git clone -q https://github.com/anas-rz/ForensiX.git /content/ForensiX; fi
%cd /content/ForensiX
!git pull -q
!pip install -q -e ".[server]"

In [ ]:
import torch
import forensix

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)
for family in ("image", "video", "audio", "audiovisual", "identity", "physics", "provenance"):
    print(f"{family:>12}: {', '.join(forensix.list_models(family))}")

In [ ]:
forensix.info("dfdet-clip")   # license, source, paper and description of any model

## 2. Sample media

- **Images:** AI-generated images (DALL-E 3, Stable Diffusion) and real NASA photos, all public domain, from Wikimedia Commons.
- **Audio:** real speech (NASA 1969, LibriVox) and synthetic speech made below with Piper TTS (LJSpeech voice, public-domain training data).
- **Video:** [SDFVD](https://data.mendeley.com/datasets/bcmkfgct2s/1), 53 real and 53 face-swapped clips, CC BY 4.0 (182 MB).
- **Provenance:** C2PA test files from [c2pa-rs](https://github.com/contentauth/c2pa-rs) (MIT/Apache-2.0): valid, AI-declared and tampered Content Credentials.

In [ ]:
import shutil
import time
import urllib.error
import urllib.request
from pathlib import Path

MEDIA = Path("/content/media")
# Wikimedia asks for a descriptive User-Agent and rate-limits bursts (HTTP 429), so retry with backoff
USER_AGENT = "ForensiX-colab/1.0 (https://github.com/anas-rz/ForensiX)"


def fetch(url, dest):
    dest = MEDIA / dest
    dest.parent.mkdir(parents=True, exist_ok=True)
    for attempt in range(6):
        if dest.exists():
            break
        try:
            with urllib.request.urlopen(urllib.request.Request(url, headers={"User-Agent": USER_AGENT})) as r:
                dest.write_bytes(r.read())
        except urllib.error.HTTPError as e:
            if e.code != 429 or attempt == 5:
                raise
            time.sleep(int(e.headers.get("Retry-After") or 0) or 5 * 2 ** attempt)
    time.sleep(1)
    return dest


COMMONS = "https://upload.wikimedia.org/wikipedia/commons/"
C2PA = "https://raw.githubusercontent.com/contentauth/c2pa-rs/main/sdk/tests/fixtures/"
fetch(COMMONS + "8/8b/An_angry_man_with_an_empty_plate%2C_hungry.jpg", "images/fake/dalle3_angry_man.jpg")
fetch(COMMONS + "f/f3/Astronaut_walking_on_the_surface_of_Mercury.jpg", "images/fake/dalle3_astronaut.jpg")
fetch(COMMONS + "9/9e/Burned_city.jpg", "images/fake/sd_burned_city.jpg")
fetch(COMMONS + "e/e4/Expedition_58_crew_portrait.jpg", "images/real/nasa_crew.jpg")
fetch(COMMONS + "a/a7/Buzz_Aldrin_%28S69-31743%29.jpg", "images/real/nasa_aldrin.jpg")
shutil.copy("tests/data/face_public_domain.jpg", MEDIA / "images/real/nasa_armstrong.jpg")
fetch(COMMONS + "d/dd/Armstrong_Small_Step.ogg", "audio/real/armstrong_small_step.ogg")
shutil.copy("tests/data/speech_cc0.flac", MEDIA / "audio/real/librivox_reading.flac")
for name in ("CA.jpg", "C.jpg", "XCA.jpg"):
    fetch(C2PA + name, f"provenance/{name}")

# SDFVD face-swap videos, checked against their published hash
!curl -sL -o /content/SDFVD.zip "https://data.mendeley.com/public-files/datasets/bcmkfgct2s/files/a9ef71a3-b625-4682-a21c-f0579e85cdd4/file_downloaded"
!echo "734a101fdfd89d5b20ac69f510d6a7cf3b588a145c1993ba2488322ae50d95a5  /content/SDFVD.zip" | sha256sum -c
!unzip -q -o /content/SDFVD.zip -d /content/sdfvd
SDFVD = Path("/content/sdfvd/SDFVD")
print(len(list((SDFVD / "videos_real").glob("*.mp4"))), "real and", len(list((SDFVD / "videos_fake").glob("*.mp4"))), "fake videos")

In [ ]:
# Synthetic speech: Piper TTS with the LJSpeech voice (its training data is public domain)
!pip install -q piper-tts
VOICE = "https://huggingface.co/rhasspy/piper-voices/resolve/main/en/en_US/ljspeech/high/en_US-ljspeech-high.onnx"
fetch(VOICE, "tts/voice.onnx"); fetch(VOICE + ".json", "tts/voice.onnx.json")
(MEDIA / "audio/fake").mkdir(parents=True, exist_ok=True)
!echo "Hi, it's me. I need you to approve the wire transfer before five o'clock today. Call me back on this number." | piper -m /content/media/tts/voice.onnx -f /content/media/audio/fake/tts_voicemail.wav
!echo "The museum will be closed on Monday for maintenance. Tickets for that day can be exchanged at the front desk." | piper -m /content/media/tts/voice.onnx -f /content/media/audio/fake/tts_announcement.wav
!find /content/media -type f -not -path "*/tts/*" | sort

## 3. Detect

Every detector has the same API: `forensix.load(name).predict(path)` returns a `Prediction` with
`fake_prob`, `threshold`, `label` and model-specific `details`. The helper below loads each model
once and puts the scores in a table.

In [ ]:
import pandas as pd
from IPython.display import display

_detectors = {}


def detector(name):
    if name not in _detectors:
        _detectors[name] = forensix.load(name, device=DEVICE)
    return _detectors[name]


def detect(paths, models):
    rows = []
    for path in paths:
        row = {"file": Path(path).name}
        for m in models:
            try:
                p = detector(m).predict(str(path))
                row[m] = f"{p.fake_prob:.3f} {p.label}"
            except Exception as e:  # e.g. no face found in the video
                row[m] = f"error: {e}"[:60]
        rows.append(row)
    return pd.DataFrame(rows)


images = sorted((MEDIA / "images").glob("*/*.jpg"))
display(detect(images, ["commfor", "univfd", "rine"]))

Audio anti-spoofing models (real historic recordings are hard: old, band-limited audio often looks synthetic to them):

In [ ]:
audio = sorted((MEDIA / "audio").glob("*/*.*"))
display(detect(audio, ["nes2net-x-itw", "aasist"]))

Video deepfake detectors on two real and two face-swapped SDFVD clips:

In [ ]:
videos = sorted((SDFVD / "videos_real").glob("*.mp4"))[:2] + sorted((SDFVD / "videos_fake").glob("*.mp4"))[:2]
display(detect(videos, ["dfdet-clip", "gend"]))

**Provenance:** evidence the file itself carries. A valid C2PA manifest declaring AI generation, a
signature broken by later edits, or a generator's invisible watermark is stronger evidence than any
detector score. Its absence proves nothing: most files carry none.

In [ ]:
for f in sorted((MEDIA / "provenance").glob("*.jpg")) + [MEDIA / "images/fake/dalle3_angry_man.jpg"]:
    report = forensix.provenance.check(str(f))
    print(f"{f.name:<22} {report['verdict']:<22} {report['summary']}")

**Your own files:** upload images, video or audio. Each is scored with the detectors for its
media type and checked for provenance evidence.

In [ ]:
from google.colab import files

DEFAULTS = {"image": ["commfor", "univfd"], "video": ["dfdet-clip"], "audio": ["nes2net-x-itw"]}
KIND = {".jpg": "image", ".jpeg": "image", ".png": "image", ".webp": "image", ".mp4": "video", ".mov": "video",
        ".webm": "video", ".mkv": "video", ".wav": "audio", ".flac": "audio", ".mp3": "audio", ".ogg": "audio", ".m4a": "audio"}
uploads = Path("/content/uploads"); uploads.mkdir(exist_ok=True)
for name, data in files.upload().items():
    path = uploads / name
    path.write_bytes(data)
    kind = KIND.get(path.suffix.lower())
    if kind is None:
        print(f"{name}: not an image, video or audio file"); continue
    display(detect([path], DEFAULTS[kind]))
    print("provenance:", forensix.provenance.check(str(path))["summary"])

## 4. Calibrate

A detector's raw score is not a probability for *your* data: thresholds trained on one dataset
rarely transfer. Calibration fits it to labelled examples of your own media and gives you:

- a **calibrated probability** of fake;
- a **threshold** that keeps false alarms at or below a target rate (`alpha`, default 5%), with a
  conformal guarantee;
- a **confidence verdict** per prediction: `["fake"]`, `["real"]`, or both when the model can't tell.

We calibrate `dfdet-clip` on 30 real and 30 fake SDFVD videos and keep the rest for evaluation.
To use your own data, zip a folder with `real/` and `fake/` subfolders (or a `manifest.csv`) and
pass `dataset="my_data.zip"`. See [Calibration](https://anas-rz.github.io/ForensiX/calibration/).

In [ ]:
real_videos = sorted((SDFVD / "videos_real").glob("*.mp4"), key=lambda p: int(p.stem[1:]))
fake_videos = sorted((SDFVD / "videos_fake").glob("*.mp4"), key=lambda p: int(p.stem[2:]))
N_CAL = 30
cal_real, held_real = real_videos[:N_CAL], real_videos[N_CAL:]
cal_fake, held_fake = fake_videos[:N_CAL], fake_videos[N_CAL:]
print(f"calibration: {len(cal_real)} real + {len(cal_fake)} fake; held out: {len(held_real)} real + {len(held_fake)} fake")

cal, report = forensix.calibrate("dfdet-clip", real_media=cal_real, fake_media=cal_fake, device=DEVICE, alpha=0.05)
Path("/content/calibrations").mkdir(exist_ok=True)
cal.save("/content/calibrations/dfdet-clip.json")

print("AUC on the calibration data:", report["auc"])
print(pd.DataFrame(report["cv"]).T.to_string())   # out-of-fold: raw threshold vs calibrated vs conformal
for w in report["warnings"]:
    print("warning:", w)

In [ ]:
calibrated = forensix.load("dfdet-clip", device=DEVICE, calibration="/content/calibrations/dfdet-clip.json")
rows = []
for path, truth in [(p, "real") for p in held_real[:3]] + [(p, "fake") for p in held_fake[:3]]:
    p = calibrated.predict(str(path))
    rows.append({"file": path.name, "truth": truth, "raw score": round(p.details["raw_score"], 3),
                 "calibrated P(fake)": round(p.fake_prob, 3), "decision": p.label,
                 "confidence": " or ".join(p.details["prediction_set"]) or "neither"})
display(pd.DataFrame(rows))

## 5. Evaluate

Never trust a detector you haven't measured on data like yours. Here we score the held-out videos,
which the calibration never saw, and compare the raw threshold with the calibrated one.
[Evaluating on your data](https://anas-rz.github.io/ForensiX/guides/evaluation/) explains the metrics and how to build a fair test set.

In [ ]:
import numpy as np
from forensix.metrics import binary_summary, compute_eer

held = [(p, False) for p in held_real] + [(p, True) for p in held_fake]
preds = [calibrated.predict(str(p)) for p, _ in held]
raw = np.array([p.details["raw_score"] for p in preds])
is_fake = np.array([f for _, f in held])
flagged = np.array([p.label == "fake" for p in preds])

summary = binary_summary(raw[is_fake], raw[~is_fake], threshold=0.5)
print(f"AUC {summary['auc']:.3f}   EER {compute_eer(raw[is_fake], raw[~is_fake])[0]:.3f}")
print(f"raw threshold 0.5:  false alarms {1 - summary['real_acc']:.1%}, fakes caught {summary['fake_acc']:.1%}")
print(f"calibrated:         false alarms {flagged[~is_fake].mean():.1%} (target 5%), fakes caught {flagged[is_fake].mean():.1%}")

To compare several models at once, use the evaluation script. It writes a results JSON
(resumable) and per-file scores. Image detectors score frames sampled from each video.

In [ ]:
import json
import os

for kind, videos in (("real", held_real), ("fake", held_fake)):
    folder = Path(f"/content/heldout/{kind}"); folder.mkdir(parents=True, exist_ok=True)
    for v in videos:
        if not (folder / v.name).exists():
            os.symlink(v, folder / v.name)
Path("/content/results").mkdir(exist_ok=True)
!python scripts/eval_video_folder.py --device {DEVICE} --real /content/heldout/real --fake /content/heldout/fake \
    --dataset "SDFVD held-out" --models dfdet-clip gend commfor univfd \
    --out /content/results/heldout.json --scores-out /content/results/heldout_scores.csv

results = json.load(open("/content/results/heldout.json"))["models"]
display(pd.DataFrame(results).T[["auc", "eer", "balanced_acc", "real_acc", "fake_acc"]].sort_values("auc", ascending=False))

## 6. Web manager and HTTP API

The same detectors run as an HTTP server with a web manager: test files, load models, calibrate on
uploaded datasets, watch live statistics, see the leaderboard and run fairness reports. Here it
runs in the background of this notebook, with the calibration from section 4 loaded.
See [HTTP server](https://anas-rz.github.io/ForensiX/server/) and [Web manager](https://anas-rz.github.io/ForensiX/ui/).

In [ ]:
import subprocess
import time
import urllib.request

config = {
    "device": DEVICE,
    "preload": ["commfor", "dfdet-clip", "nes2net-x-itw"],
    "calibrations": {"dfdet-clip": "/content/calibrations/dfdet-clip.json"},
    "calibration_dir": "/content/forensix_data",   # uploaded datasets, calibrations and reports
}
Path("/content/server.json").write_text(json.dumps(config))
log = open("/content/server.log", "w")
server = subprocess.Popen(["python", "-m", "forensix.server", "--config", "/content/server.json", "--port", "8000"],
                          stdout=log, stderr=subprocess.STDOUT)
for _ in range(300):
    try:
        print(urllib.request.urlopen("http://localhost:8000/ready").read().decode()); break
    except Exception:
        time.sleep(2)
else:
    print(open("/content/server.log").read()[-3000:])

In [ ]:
from google.colab import output

output.serve_kernel_port_as_window(8000, path="/ui/", anchor_text="Open the ForensiX web manager")
output.serve_kernel_port_as_window(8000, path="/docs", anchor_text="Open the interactive API docs")

The HTTP API, from Python (or `curl -F file=@photo.jpg http://localhost:8000/v1/detect`):

In [ ]:
import requests

with open(MEDIA / "images/fake/dalle3_angry_man.jpg", "rb") as f:
    r = requests.post("http://localhost:8000/v1/detect", files={"file": f}).json()
for model, res in r["results"].items():
    print(f"{model:<16}", res.get("label") or res["provenance"]["verdict"], res.get("score"))

**Optional: fairness report.** Do detectors raise more false alarms for some skin tones, ages or
genders? This builds a balanced audit set of 420 real faces from
[FairFace](https://github.com/joojs/fairface) (CC BY 4.0, 237 MB download), uploads it, and
reports false-alarm rates per group for `commfor`. Open the **Fairness** page of the web manager
to see the charts. See [Fairness reports](https://anas-rz.github.io/ForensiX/fairness/).

In [ ]:
!curl -sL -o /content/fairface_val.parquet "https://huggingface.co/datasets/HuggingFaceM4/FairFace/resolve/54d573cdb8b5af490ba8da9da2799628f6e5c496/1.25/validation-00000-of-00001-09e3e67bb00ab4ec.parquet"
!python scripts/build_fairface_set.py --parquet /content/fairface_val.parquet --out /content/fairface/fairface_audit
!cd /content/fairface && zip -qr fairface_audit.zip fairface_audit

with open("/content/fairface/fairface_audit.zip", "rb") as f:
    dataset = requests.post("http://localhost:8000/v1/datasets", files={"file": f}, data={"name": "fairface-audit"}).json()
job = requests.post("http://localhost:8000/v1/fairness", json={"dataset": dataset["id"], "models": ["commfor"]}).json()
while (status := requests.get(f"http://localhost:8000/v1/jobs/{job['id']}").json())["status"] in ("queued", "running"):
    time.sleep(10)
print(status["status"], status.get("error") or "")
print(requests.get(f"http://localhost:8000/v1/fairness/{job['id']}", params={"format": "markdown"}).text[:2500])

## 7. Keep your results

Colab deletes `/content` when the runtime ends. Copy calibrations, results and the server's data
to Google Drive.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/forensix
!cp -r /content/calibrations /content/results /content/drive/MyDrive/forensix/
!cp -r /content/forensix_data /content/drive/MyDrive/forensix/ 2>/dev/null || true
!ls -R /content/drive/MyDrive/forensix | head -20

## Next steps

- [Quickstart](https://anas-rz.github.io/ForensiX/getting-started/) and [Recipes](https://anas-rz.github.io/ForensiX/recipes/): working code for common tasks.
- [Choosing detectors](https://anas-rz.github.io/ForensiX/guides/choosing-detectors/): which detector catches which manipulation.
- [Leaderboard](https://anas-rz.github.io/ForensiX/reference/leaderboard/): every model on commercially usable benchmarks.
- [Working in Google Colab](https://anas-rz.github.io/ForensiX/guides/colab/): GPUs, Drive, GitHub and long runs.
- [Docker, Kubernetes and cloud](https://anas-rz.github.io/ForensiX/cloud/): run the server in production.